# Image Classification with PyTorch Hub (Step-by-Step)
This notebook demonstrates how to use PyTorch Hub to load a pre-trained model and classify an image, following industry best practices.

## Step 1: Import Required Libraries

In [ ]:
import torch
import torchvision.transforms as transforms
from PIL import Image
import requests
from io import BytesIO

## Step 2: Load a Pre-trained Model from PyTorch Hub
We'll use ResNet-50 for this example.

In [ ]:
model = torch.hub.load('pytorch/vision:v0.10.0', 'resnet50', pretrained=True)
model.eval()

## Step 3: Download and Prepare an Input Image

In [ ]:
# Example image (dog)
url = 'https://github.com/pytorch/hub/raw/master/images/dog.jpg'
response = requests.get(url)
img = Image.open(BytesIO(response.content))
img

## Step 4: Preprocess the Image

In [ ]:
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])
input_tensor = preprocess(img)
input_batch = input_tensor.unsqueeze(0)  # create a mini-batch as expected by the model

## Step 5: Run Inference

In [ ]:
with torch.no_grad():
    output = model(input_batch)
# The output has unnormalized scores. To get probabilities, you can run a softmax on it.
probabilities = torch.nn.functional.softmax(output[0], dim=0)

## Step 6: Get Top-5 Predictions

In [ ]:
# Download ImageNet labels
labels_url = 'https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt'
labels = requests.get(labels_url).text.splitlines()

# Show top 5 predictions
_, indices = torch.topk(probabilities, 5)
for idx in indices:
    print(labels[idx], probabilities[idx].item())